# imports

In [ ]:
install.packages("ggtext", repos = "https://cloud.r-project.org")

In [ ]:
library(dplyr)
library(MatchIt)
library(readr)
library(stringr)
library(tableone)
library(tibble)
library(tidyr)
library(ggplot2)
library(cobalt)
library(purrr)
library(cluster)
library(sandwich)
library(lmtest)
library(sensemakr)
library(EValue)
library(rbounds)
library(patchwork)
library(brglm2)
library(ggtext)
library(survival)
library(marginaleffects)
library(lubridate)


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)

# ─────────────────────────────────────────────
# Color palette
# ─────────────────────────────────────────────
COL_BLUE <- "#2D6A9F"   # Steel blue (treatment)
COL_GOLD <- "#E09B3D"   # Warm gold (control)
COL_GRAY <- "#B0B0B0"   # Light gray (not significant)
COL_ROSE <- "#C45B5B"   # Muted rose (before matching)

# ─────────────────────────────────────────────
# Shared helpers
# ─────────────────────────────────────────────


# Variable name translations for love plots (column names + Hebrew factor levels)
love_var_translate <- function(nms) {
  translations <- c(
    "Criminal_record"          = "Criminal Record",
    "marital_status"           = "Marital Status",
    "ethnic"                   = "ethnic",
    "judge_ethnic"             = "Judge Ethnicity",
    "regret"                   = "Regret",
    "confession"               = "Guilty Plea",
    "min_punishment"           = "Lower Bounds",
    "max_punishment"           = "Upper Bounds",
    "district"                 = "District",
    "age"                      = "Age",
    "year"                     = "Year",
    "Court"                    = "Court Type",
    "weapon_motive"            = "Weapon Motive",
    "weapon_storage"           = "Weapon Storage",
    "weapon_use"               = "Weapon Use",
    "actual_damage"            = "Actual Damage",
    "potential_damage"         = "Potential Damage",
    "premeditation"            = "Premeditation",
    "escape_attempt"           = "Escape Attempt",
    "has_independent_offenses" = "Independent Offenses",
    "undercover_sale"          = "Undercover Sale",
    "handgun"                  = "Handgun",
    "submachine_gun"           = "Submachine Gun",
    "assault_rifle"            = "Assault Rifle",
    "sniper_rifle"             = "Sniper Rifle",
    "hunting_rifle"            = "Hunting Rifle",
    "improvised_firearm"       = "Improvised Firearm",
    "grenade"                  = "Grenade",
    "explosive_charge"         = "Explosive Charge",
    "molotov"                  = "Molotov",
    "anti_tank_missile"        = "Anti-Tank Missile",
    "ammunition_only"          = "Ammunition Only",
    "weapon_parts"             = "Weapon Parts",
    "unidentified_firearm"     = "Unidentified Firearm",
    "magazines"                = "Magazines",
    "bullets"                  = "Bullets",
    "לא ידוע"               = "Unknown",
    "ללא עבר פלילי" = "No Criminal Record",
    "עבר פלילי כבד" = "Severe Criminal Record",
    "עבר פלילי קל"   = "Minor Criminal Record",
    "גרוש"                 = "Divorced",
    "נשוי"                 = "Married",
    "רווק"                 = "Single",
    "מחוזי"           = "District Court",
    "שלום"                 = "Magistrate Court"
  )
  for (old in names(translations)) {
    nms <- gsub(old, translations[old], nms, fixed = TRUE)
  }
  nms
}


# Academic color palette (grayscale-friendly)
theme_academic <- function() {
  theme_gray() +
    theme(
      text = element_text(size = 11),
      plot.title = element_text(size = 12, face = "bold", hjust = 0.5),
      plot.background = element_rect(fill = "white", color = NA),
      axis.title = element_text(size = 11),
      axis.text = element_text(size = 10, color = "black"),
      legend.position = "top",
      legend.title = element_blank()
    )
}

# ─────────────────────────────────────────────
#  Propensity Score Distribution
# ─────────────────────────────────────────────

# Alternative: Boxplot version
plot_ps_distribution_boxplot <- function(matched_df, method_label = "", 
                                        file_path = NULL, width = 8, height = 4) {
  
  if (!("distance" %in% names(matched_df)) || !("treated" %in% names(matched_df))) {
    message("No distance/treated columns found.")
    return(invisible(NULL))
  }
  
  df_plot <- matched_df %>%
    filter(!is.na(distance)) %>%
    mutate(group = ifelse(treated == 1, "Treatment", "Control"))
  
  p <- ggplot(df_plot, aes(x = group, y = distance, fill = group)) +
    geom_boxplot(outlier.size = 1) +
    scale_fill_manual(values = c("Control" = COL_GOLD, "Treatment" = COL_BLUE)) +
    labs(title = paste("Propensity Score Distribution —", method_label),
         x = NULL, y = "Propensity Score") +
    theme_academic() +
    theme(legend.position = "none")
  
  # Save if file path provided
  if (!is.null(file_path)) {
    ggsave(file_path, plot = p, width = width, height = height, dpi = 300, bg = "white")
  }
  
  return(p)
}



# Main Analaysis 

In [ ]:
# =============================================================================
# run_matching_weapons.R
# PSM Pipeline — Weapons (mirrors the drug pipeline; only variables adapted)
# =============================================================================

# =============================================================================
# WEAPON COLUMN DEFINITIONS
# =============================================================================
# All weapon columns are COUNTS (log-transformed), EXCEPT ammunition_only,
# which is a true 0/1 flag. A count column may happen to contain only 0/1 in a
# given dataset — it is STILL a count and still gets logged. Classification is
# by semantics, not by observed values.
WEAPON_COUNT_COLS <- c(
  "handgun", "submachine_gun", "assault_rifle", "sniper_rifle", "hunting_rifle",
  "improvised_firearm", "grenade", "explosive_charge", "molotov",
  "anti_tank_missile", "weapon_parts", "unidentified_firearm",
  "magazines", "bullets"
)
WEAPON_BINARY_COLS <- c("ammunition_only")

# =============================================================================
# SHARED HELPERS
# =============================================================================

is_binary_01 <- function(x) {
  if (is.data.frame(x) || is.list(x)) return(FALSE)
  ux <- unique(x[!is.na(x)])
  length(ux) == 2 && all(as.numeric(sort(ux)) == c(0, 1))
}

gelman_scale_2sd <- function(x) {
  m <- mean(x, na.rm = TRUE)
  s <- sd(x,   na.rm = TRUE)
  if (is.na(s) || s == 0) return(x)
  (x - m) / (2 * s)
}

apply_gelman_scaling <- function(df, covars) {
  for (v in covars) {
    x <- df[[v]]
    df[[v]] <- if (is.factor(x) || is.character(x)) {
      as.factor(x)
    } else if (is.logical(x)) {
      as.integer(x)
    } else if (is.numeric(x) || is.integer(x)) {
      if (is_binary_01(x)) as.integer(x) else gelman_scale_2sd(as.numeric(x))
    } else {
      as.factor(x)
    }
  }
  df
}

compute_cohens_d <- function(x_t, x_c) {
  n_t <- sum(!is.na(x_t)); n_c <- sum(!is.na(x_c))
  if (n_t < 2 || n_c < 2) return(NA_real_)
  sd_pooled <- sqrt(
    ((n_t - 1) * var(x_t, na.rm = TRUE) +
     (n_c - 1) * var(x_c, na.rm = TRUE)) / (n_t + n_c - 2)
  )
  if (!is.finite(sd_pooled) || sd_pooled == 0) return(NA_real_)
  (mean(x_t, na.rm = TRUE) - mean(x_c, na.rm = TRUE)) / sd_pooled
}

# 1:1 nearest, no replacement → all weights == 1, so unweighted == weighted.
fit_model <- function(form, data, is_binary) {
  if (is_binary) glm(form, data = data, family = binomial())
  else           lm( form, data = data)
}

add_pair_id <- function(df) {
  if ("subclass" %in% names(df)) rename(df, pair_id = subclass)
  else mutate(df, pair_id = as.character(row_number()))
}

sig_label <- function(p) {
  case_when(p <= 0.001 ~ "***", p <= 0.01 ~ "**", p <= 0.05 ~ "*", TRUE ~ "")
}

extract_or <- function(is_binary, robust_coef, ci_robust) {
  if (!is_binary) return(list(val = NA_real_, lo = NA_real_, hi = NA_real_))
  list(val = exp(robust_coef["treated", 1]),
       lo  = exp(ci_robust["treated", 1]),
       hi  = exp(ci_robust["treated", 2]))
}

# Log / un-log helpers for weapon count columns
log_weapon_counts <- function(df) {
  cols <- intersect(WEAPON_COUNT_COLS, names(df))
  if (length(cols)) df <- df %>%
    mutate(across(all_of(cols), ~ log1p(pmax(as.numeric(.x), 0))))
  df
}
unlog_weapon_counts <- function(df) {
  cols <- intersect(WEAPON_COUNT_COLS, names(df))
  if (length(cols)) df <- df %>%
    mutate(across(all_of(cols), ~ expm1(.x)))
  df
}

# =============================================================================
# MAIN FUNCTION
# =============================================================================

run_matching <- function(
    df, group_col, targets_cols, output_dir, treatment, control,
    caliper_q       = 0.9,
    caliper_ps_sd   = 0.2,
    match_mode      = c("gower", "logit", "exact_cem", "no_caliper", "custom"),
    all_covars      = NULL,
    fact_covars     = NULL,
    demo_covars     = NULL,
    gower_demo_weight = 1,
    gower_fact_weight = 4,
    only_fact_vars  = FALSE,
    binary_targets  = NULL,
    interaction_var = "ethnic",
    war_interaction = FALSE,
    bonferroni_k    = 1
) {
  match_mode <- match.arg(match_mode)

  # --- 1. COVARIATE SETUP ---
  base_covars <- if (only_fact_vars) fact_covars else all_covars
  base_covars <- setdiff(base_covars, group_col)

  # --- 2. DATA PREPARATION ---
  df_before <- df %>%
    filter(.data[[group_col]] %in% c(treatment, control)) %>%
    drop_na(any_of(c(group_col, base_covars, targets_cols))) %>%
    mutate(treated = as.integer(.data[[group_col]] == treatment))

  df_group <- log_weapon_counts(df_before)

  # Factorize categoricals (keep year numeric)
  for (v in base_covars) {
    if (!v %in% names(df_group) || v == "year") next
    if (v %in% WEAPON_COUNT_COLS) next          
    if (is.character(df_group[[v]]) || length(unique(df_group[[v]])) < 10) {
      df_group[[v]]  <- as.factor(df_group[[v]])
      df_before[[v]] <- as.factor(df_before[[v]])
    }
  }

  covars <- intersect(base_covars, names(df_group))
  for (v in covars)
    if (is.factor(df_group[[v]])) df_group[[v]] <- droplevels(df_group[[v]])

  is_constant <- function(x)
    if (is.factor(x)) nlevels(x) < 2 else length(unique(x[!is.na(x)])) < 2
  drop_const <- covars[sapply(covars, function(v) is_constant(df_group[[v]]))]
  if (length(drop_const) > 0) {
    message("  Dropping constant covars: ", paste(drop_const, collapse = ", "))
    covars <- setdiff(covars, drop_const)
  }

  if ("year" %in% covars) {
    df_group  <- df_group  %>% mutate(year = year - min(year, na.rm = TRUE))
    df_before <- df_before %>% mutate(year = year - min(year, na.rm = TRUE))
  }

  form <- as.formula(
    paste("treated ~", paste(paste0("`", covars, "`"), collapse = " + "))
  )

  # --- 3. OUTPUT DIRECTORY ---
  match_mode_prefix <- if (only_fact_vars) "fact_match_" else "match_"
  output_dir <- file.path(output_dir, paste0(match_mode_prefix, match_mode))
  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

  # --- 4. MATCHING ---
  use_gower <- (match_mode == "gower") && (!only_fact_vars)

  matchit_obj <- if (use_gower) {
    run_gower_matching(df_group, covars, demo_covars, group_col, form,
                       gower_demo_weight, gower_fact_weight, caliper_q)
  } else if (match_mode %in% c("logit", "custom")) {
    matchit(form, data = df_group, method = "nearest", distance = "logit",
            caliper = caliper_ps_sd, std.caliper = TRUE,
            replace = FALSE, ratio = 1)
  } else if (match_mode == "no_caliper") {
    matchit(form, data = df_group, method = "nearest", distance = "logit",
            std.caliper = TRUE, replace = FALSE, ratio = 1)
  } else {
    # exact_cem: nearest + exact + Mahalanobis on weapon counts
    mah_vars <- intersect(WEAPON_COUNT_COLS, covars)
    mah_form <- as.formula(paste("~", paste(paste0("`", mah_vars, "`"), collapse = " + ")))
    matchit(form, data = df_group, method = "nearest", distance = "logit",
            exact   = intersect(c("actual_damage", "premeditation", "escape_attempt",
                                  "has_independent_offenses", "undercover_sale"),
                                names(df_group)),
            mahvars = mah_form,
            caliper = caliper_ps_sd, std.caliper = TRUE,
            replace = FALSE, ratio = 1)
  }

  # --- 5. EXTRACT MATCHED DATA ---
  # Propensity scores for every unit (before matching) — used for the SM overlap figures
  if (is.numeric(matchit_obj$distance) && is.null(dim(matchit_obj$distance))) {
    write.csv(data.frame(treated  = as.integer(matchit_obj$treat),
                         distance = matchit_obj$distance,
                         matched  = as.integer(matchit_obj$weights > 0)),
              file.path(output_dir, "ps_all.csv"), row.names = FALSE)
  }

  matched_long <- get_matches(matchit_obj, data = df_group)
  if (!nrow(matched_long)) return(NULL)
  matched_long <- matched_long %>% add_pair_id() %>%
    arrange(pair_id, desc(treated)) %>%
    unlog_weapon_counts()
  write.csv(matched_long, file.path(output_dir, "matched_long.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  matched_df <- match.data(matchit_obj)
  if (!nrow(matched_df)) return(NULL)
  matched_df <- matched_df %>% add_pair_id() %>% unlog_weapon_counts()
  write.csv(matched_df, file.path(output_dir, "matched.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  # Re-log for modeling
  matched_df <- log_weapon_counts(matched_df)

  # --- 6. BALANCE TABLE ---
  balance <- tryCatch({
    s <- summary(matchit_obj)
    merge(tibble::rownames_to_column(as.data.frame(s$sum.all),     "Covariate"),
          tibble::rownames_to_column(as.data.frame(s$sum.matched), "Covariate"),
          by = "Covariate", suffixes = c("_before", "_after"))
  }, error = function(e) { message("Balance summary failed: ", conditionMessage(e)); NULL })
  if (!is.null(balance))
    write.csv(balance, file.path(output_dir, "balance_table.csv"),
              fileEncoding = "UTF-8", row.names = FALSE)

  # --- 7. PROPENSITY SCORE MODEL ---
  ps_coef_df <- fit_propensity_model(df_group, covars, output_dir)

  # --- 8. OUTCOME ANALYSIS ---
  result_df <- compute_outcome_results(
    matched_df, matched_long, targets_cols, binary_targets,
    use_sensitivity = TRUE, bonferroni_k = bonferroni_k
  )
  write.csv(result_df, file.path(output_dir, "results.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  result_before_df <- compute_outcome_results_before(
    df_before, targets_cols, binary_targets
  )
  write.csv(result_before_df, file.path(output_dir, "results_before_matching.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)

  # --- 9. PLOTS ---
  ps_box_plot <- plot_ps_distribution_boxplot(
    matched_df = matched_df, method_label = match_mode,
    file_path = file.path(output_dir, "ps_boxplot.png")
  )

  # --- 10. FULL REGRESSION MODELS ---
  full_coef_df <- fit_full_model_after(
    matched_df, targets_cols, binary_targets, covars, output_dir,
    group_col, interaction_var, war_interaction = war_interaction
  )
  full_coef_before_df <- fit_full_model_before(
    df_before, targets_cols, binary_targets, covars, output_dir,
    group_col, interaction_var
  )

  # --- 11. COMPARISON TABLE & LOVE PLOT ---
  comparison_formatted <- build_comparison_table(
    df_before, matched_df, targets_cols, binary_targets, output_dir
  )
  love_plot <- create_love_plot(matchit_obj, output_dir)

  # --- 12. RETURN ---
  list(
    matched_df = matched_df, matched_long = matched_long,
    results = result_df, comparison_table = comparison_formatted,
    balance = balance, love_plot = love_plot, matchit_obj = matchit_obj,
    ps_coef = ps_coef_df, df_before = df_before,
    full_coef_after = full_coef_df, full_coef_before = full_coef_before_df,
    ps_box_plot = ps_box_plot
  )
}


# =============================================================================
# SUB-FUNCTIONS
# =============================================================================

# --- Gower Distance Matching ---
run_gower_matching <- function(df_group, covars, demo_covars, group_col,
                               form, gower_demo_weight, gower_fact_weight,
                               caliper_q) {
  X <- df_group[, covars, drop = FALSE]
  demo_covars <- intersect(demo_covars, covars)

  w <- setNames(rep(gower_fact_weight, length(covars)), covars)
  w[demo_covars] <- gower_demo_weight

  # Detect binary vars by their *levels*, not assuming {0,1} — weapon counts
  # that were log-transformed have 2 distinct values {0, 0.693}, so use the
  # actual sorted unique values as factor levels.
  binary_vars <- names(X)[sapply(X, function(x)
    is.numeric(x) && length(unique(x[!is.na(x)])) == 2)]
  for (v in binary_vars) {
    lv <- sort(unique(X[[v]][!is.na(X[[v]])]))
    X[[v]] <- factor(X[[v]], levels = lv)
  }

  types <- list(nominal = names(X)[sapply(X, is.factor)])
  D <- as.matrix(daisy(X, metric = "gower", weights = w, type = types))

  idx_t <- which(df_group$treated == 1)
  idx_c <- which(df_group$treated == 0)
  dist_mat <- D[idx_t, idx_c, drop = FALSE]

  nn_distances <- apply(dist_mat, 1, function(r) {
    r <- r[is.finite(r)]
    if (length(r) == 0) NA_real_ else min(r)
  })
  caliper_val <- unname(quantile(nn_distances, probs = caliper_q, na.rm = TRUE))

  matchit(form, data = df_group, method = "nearest", distance = dist_mat,
          caliper = caliper_val, std.caliper = FALSE,
          replace = FALSE, ratio = 1)
}

# --- Propensity Score Model ---
fit_propensity_model <- function(df_group, covars, output_dir) {
  ps_df <- df_group[, c("treated", covars), drop = FALSE]
  ps_df$treated <- case_when(
    is.logical(ps_df$treated) ~ as.integer(ps_df$treated),
    is.factor(ps_df$treated)  ~ as.integer(ps_df$treated) - 1,
    TRUE                      ~ as.numeric(ps_df$treated)
  )
  ps_df <- apply_gelman_scaling(ps_df, covars)

  ps_fit <- tryCatch(glm(treated ~ ., data = ps_df, family = binomial()),
                     error = function(e) NULL)
  if (is.null(ps_fit)) return(NULL)

  ps_coef_df <- as.data.frame(summary(ps_fit)$coefficients) %>%
    tibble::rownames_to_column("Covariate") %>%
    dplyr::rename(Std_Error = `Std. Error`, z_value = `z value`, p_value = `Pr(>|z|)`) %>%
    dplyr::mutate(Significance = cut(p_value, c(-Inf, 0.001, 0.01, 0.05, Inf),
                                     labels = c("***", "**", "*", ""))) %>%
    arrange(p_value)
  write.csv(ps_coef_df, file.path(output_dir, "model_coefficients.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  ps_coef_df
}

# --- Outcome Analysis — After Matching ---
compute_outcome_results <- function(matched_df, matched_long, targets_cols,
                                    binary_targets, use_sensitivity = TRUE,
                                    bonferroni_k = 1) {
  bind_rows(lapply(targets_cols, function(col) {

    is_binary  <- col %in% binary_targets
    model_form <- as.formula(paste(col, "~ treated"))

    # Complete-separation guard for binary outcomes
    if (is_binary) {
      contingency <- table(matched_df$treated, matched_df[[col]])
      if (any(contingency == 0)) {
        cat("  WARNING: Complete separation in", col, "— OR unreliable, returning NA\n")
        m_t <- mean(matched_df[[col]][matched_df$treated == 1], na.rm = TRUE)
        m_c <- mean(matched_df[[col]][matched_df$treated == 0], na.rm = TRUE)
        return(tibble(
          outcome = col, is_binary = TRUE,
          mean_treatment = m_t, mean_control = m_c, diff = m_t - m_c,
          odds_ratio = NA_real_, or_ci_low = NA_real_, or_ci_high = NA_real_,
          std_error = NA_real_, ci_low = NA_real_, ci_high = NA_real_,
          d_cohens = NA_real_,
          p_value = NA_real_, p_bonferroni = NA_real_,
          rosenbaum_gamma = NA_real_, e_value_point = NA_real_, eval_ci = NA_real_,
          n_treatment = sum(matched_df$treated == 1),
          n_control   = sum(matched_df$treated == 0),
          n_treatment_unique = sum(matched_df$treated == 1),
          n_control_unique   = sum(matched_df$treated == 0),
          sd_treatment = sd(matched_df[[col]][matched_df$treated == 1], na.rm = TRUE),
          sd_control   = sd(matched_df[[col]][matched_df$treated == 0], na.rm = TRUE),
          rmse = NA_real_
        ))
      }
    }

    fit      <- fit_model(model_form, matched_df, is_binary)
    rmse_val <- if (!is_binary) sqrt(mean(residuals(fit)^2, na.rm = TRUE)) else NA_real_

    robust_coef <- coeftest(fit, vcov = vcovHC, type = "HC3")
    ci_robust   <- lmtest::coefci(fit, vcov = vcovHC, type = "HC3", level = 0.95)
    or          <- extract_or(is_binary, robust_coef, ci_robust)

    means <- matched_df %>% group_by(treated) %>%
      summarise(m = mean(.data[[col]], na.rm = TRUE), .groups = "drop")

    x_t <- matched_df[[col]][matched_df$treated == 1]
    x_c <- matched_df[[col]][matched_df$treated == 0]
    d_value <- if (is_binary) {
      2 * asin(sqrt(mean(x_t, na.rm = TRUE))) - 2 * asin(sqrt(mean(x_c, na.rm = TRUE)))
    } else {
      compute_cohens_d(x_t, x_c)
    }

    gamma <- if (use_sensitivity) compute_rosenbaum_bounds(matched_long, col, is_binary)
             else NA_real_
    ev <- compute_evalues(robust_coef, ci_robust, matched_df, col,
                          is_binary, or$val, or$lo, or$hi)

    if (is_binary) {
      ame <- avg_comparisons(fit, variables = "treated",
                             vcov = sandwich::vcovHC(fit, type = "HC3"))
      diff_val <- ame$estimate; se_val <- ame$std.error
      ci_low_val <- ame$conf.low; ci_high_val <- ame$conf.high; p_val <- ame$p.value
    } else {
      diff_val <- robust_coef["treated", 1]; se_val <- robust_coef["treated", 2]
      ci_low_val <- ci_robust["treated", 1]; ci_high_val <- ci_robust["treated", 2]
      p_val <- robust_coef["treated", 4]
    }

    tibble(
      outcome = col, is_binary = is_binary,
      mean_treatment = means$m[means$treated == 1],
      mean_control   = means$m[means$treated == 0],
      diff = diff_val, odds_ratio = or$val,
      or_ci_low = or$lo, or_ci_high = or$hi,
      std_error = se_val, ci_low = ci_low_val, ci_high = ci_high_val,
      d_cohens = d_value,
      p_value = p_val, p_bonferroni = pmin(p_val * bonferroni_k, 1),
      rosenbaum_gamma = gamma,
      e_value_point = ev$point, eval_ci = ev$ci,
      n_treatment = sum(matched_df$treated == 1),
      n_control   = sum(matched_df$treated == 0),
      n_treatment_unique = sum(matched_df$treated == 1),
      n_control_unique   = sum(matched_df$treated == 0),
      sd_treatment = sd(x_t, na.rm = TRUE), sd_control = sd(x_c, na.rm = TRUE),
      rmse = rmse_val
    )
  }))
}

# --- Rosenbaum Bounds ---
compute_rosenbaum_bounds <- function(matched_long, col, is_binary) {
  pairs_data <- matched_long %>%
    select(pair_id, treated, all_of(col)) %>%
    group_by(pair_id, treated) %>%
    summarise(across(all_of(col), first), .groups = "drop") %>%
    pivot_wider(names_from = treated, values_from = all_of(col),
                names_prefix = "treat_") %>%
    drop_na()
  if (nrow(pairs_data) <= 10) return(NA_real_)

  gamma_tipping_point <- NA_real_
  pdf(file = NULL); on.exit(dev.off(), add = TRUE)
  try({
    rb_results <- if (is_binary) {
      n_plus  <- sum(pairs_data$treat_1 == 1 & pairs_data$treat_0 == 0)
      n_minus <- sum(pairs_data$treat_1 == 0 & pairs_data$treat_0 == 1)
      rbounds::binarysens(n_plus, n_minus, Gamma = 12, GammaInc = 0.1)
    } else {
      rbounds::psens(pairs_data$treat_1, pairs_data$treat_0, Gamma = 12, GammaInc = 0.1)
    }
    bounds_df   <- rb_results$bounds
    tipping_row <- which(bounds_df[, "Upper bound"] > 0.05)[1]
    gamma_tipping_point <- if (!is.na(tipping_row)) bounds_df[tipping_row, "Gamma"] else Inf
  }, silent = TRUE)
  gamma_tipping_point
}

# --- E-Values ---
compute_evalues <- function(robust_coef, ci_robust, matched_df, col,
                            is_binary, or_val, or_ci_low, or_ci_high) {
  eval_point <- NA_real_; eval_ci <- NA_real_
  suppressMessages(try({
    ev_obj <- if (is_binary) {
      prev <- mean(matched_df[[col]], na.rm = TRUE)
      EValue::evalues.OR(est = or_val, lo = or_ci_low, hi = or_ci_high,
                         rare = (prev < 0.15))
    } else {
      EValue::evalues.OLS(est = robust_coef["treated", 1],
                          se = robust_coef["treated", 2],
                          sd = sd(matched_df[[col]], na.rm = TRUE))
    }
    eval_point <- ev_obj[2, "point"]; eval_ci <- ev_obj[2, "lower"]
  }, silent = TRUE))
  list(point = eval_point, ci = eval_ci)
}

# --- Outcome Analysis — Before Matching ---
compute_outcome_results_before <- function(df_before, targets_cols, binary_targets) {
  bind_rows(lapply(targets_cols, function(col) {
    is_binary   <- col %in% binary_targets
    model_form  <- as.formula(paste(col, "~ treated"))
    fit         <- fit_model(model_form, df_before, is_binary)
    rmse_val    <- if (!is_binary) sqrt(mean(residuals(fit)^2, na.rm = TRUE)) else NA_real_
    robust_coef <- coeftest(fit, vcov = vcovHC, type = "HC3")
    ci_robust   <- lmtest::coefci(fit, vcov = vcovHC, type = "HC3", level = 0.95)
    or          <- extract_or(is_binary, robust_coef, ci_robust)
    means <- df_before %>% group_by(treated) %>%
      summarise(mean = mean(.data[[col]], na.rm = TRUE), .groups = "drop")
    x_t <- df_before[[col]][df_before$treated == 1]
    x_c <- df_before[[col]][df_before$treated == 0]
    tibble(
      outcome = col, is_binary = is_binary,
      mean_treatment = means$mean[means$treated == 1],
      mean_control   = means$mean[means$treated == 0],
      diff = robust_coef["treated", 1], odds_ratio = or$val,
      or_ci_low = or$lo, or_ci_high = or$hi,
      std_error = robust_coef["treated", 2],
      ci_low = ci_robust["treated", 1], ci_high = ci_robust["treated", 2],
      d_cohens = if (is_binary) NA_real_ else compute_cohens_d(x_t, x_c),
      p_value = robust_coef["treated", 4],
      sd_treatment = sd(x_t, na.rm = TRUE), sd_control = sd(x_c, na.rm = TRUE),
      n_treatment = sum(!is.na(x_t)), n_control = sum(!is.na(x_c)),
      rmse = rmse_val
    )
  }))
}

# --- Full Regression Model — After Matching ---
fit_full_model_after <- function(matched_df, targets_cols, binary_targets,
                                 covars, output_dir, group_col, interaction_var,
                                 war_interaction = FALSE) {
  full_coef_df <- bind_rows(lapply(targets_cols, function(col) {
    valid_covars <- Filter(function(v) {
      if (!v %in% names(matched_df)) return(FALSE)
      x <- matched_df[[v]]
      if (is.factor(x) || is.character(x)) nlevels(droplevels(as.factor(x))) >= 2
      else var(x, na.rm = TRUE) > 0
    }, covars)

    covars_for_model <- if (group_col == "ethnic")
      valid_covars[valid_covars != interaction_var] else valid_covars

    use_war_ix <- war_interaction &&
                  "war_period" %in% names(matched_df) &&
                  !identical(matched_df$war_period, matched_df$treated)

    rhs <- if (length(covars_for_model) == 0) {
      "treated"
    } else if (use_war_ix) {
      paste0("treated * war_period + ",
             paste(setdiff(covars_for_model, "war_period"), collapse = " + "))
    } else {
      paste0("treated + ", paste(covars_for_model, collapse = " + "))
    }

    model_form <- as.formula(paste(col, "~", rhs))
    is_binary  <- col %in% binary_targets
    fit <- tryCatch(fit_model(model_form, matched_df, is_binary), error = function(e) NULL)
    if (is.null(fit)) return(NULL)
    extract_model_coefs(fit, matched_df, col, is_binary, cluster_var = "pair_id")
  }))
  write.csv(full_coef_df, file.path(output_dir, "full_model_coefficients.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  full_coef_df
}

# --- Full Regression Model — Before Matching ---
fit_full_model_before <- function(df_before, targets_cols, binary_targets,
                                  covars, output_dir, group_col, interaction_var) {
  full_coef_df <- bind_rows(lapply(targets_cols, function(col) {
    is_binary <- col %in% binary_targets
    covars_for_model <- if (group_col == "ethnic")
      covars[covars != interaction_var] else covars
    model_form <- as.formula(
      paste(col, "~ treated +", paste(paste0("`", covars_for_model, "`"), collapse = " + "))
    )
    fit <- fit_model(model_form, df_before, is_binary)
    robust_vcov <- tryCatch({
      vc <- vcovHC(fit, type = "HC3")
      if (any(is.na(diag(vc)))) vcov(fit) else vc
    }, error = function(e) vcov(fit))
    extract_model_coefs_from_vcov(fit, robust_vcov, col, is_binary)
  }))
  write.csv(full_coef_df, file.path(output_dir, "full_model_coefficients_before.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  full_coef_df
}

# --- Extract Model Coefficients ---
extract_model_coefs <- function(fit, data, outcome_col, is_binary, cluster_var = NULL) {
  robust_vcov <- if (!is.null(cluster_var)) vcovCL(fit, cluster = data[[cluster_var]])
                 else vcovHC(fit, type = "HC3")
  extract_model_coefs_from_vcov(fit, robust_vcov, outcome_col, is_binary)
}

extract_model_coefs_from_vcov <- function(fit, robust_vcov, outcome_col, is_binary) {
  robust_coef   <- coeftest(fit, vcov = robust_vcov)
  ci_robust     <- coefci(fit, vcov. = robust_vcov, level = 0.95)
  model_summary <- summary(fit)
  rmse          <- sqrt(mean(residuals(fit)^2))

  if (is_binary) {
    r_squared <- 1 - model_summary$deviance / model_summary$null.deviance
    adj_r_squared <- NA_real_; f_stat <- NA_real_; f_p_value <- NA_real_
  } else {
    r_squared <- model_summary$r.squared
    adj_r_squared <- model_summary$adj.r.squared
    if (!is.null(model_summary$fstatistic)) {
      f_stat    <- model_summary$fstatistic[1]
      f_p_value <- pf(f_stat, model_summary$fstatistic[2],
                      model_summary$fstatistic[3], lower.tail = FALSE)
    } else { f_stat <- NA_real_; f_p_value <- NA_real_ }
  }

  data.frame(
    term = rownames(robust_coef), B = robust_coef[, 1], SE = robust_coef[, 2],
    t_value = robust_coef[, 3], p_value = robust_coef[, 4],
    CI_low = ci_robust[, 1], CI_high = ci_robust[, 2], RMSE = rmse
  ) %>%
    mutate(outcome = outcome_col, R_squared = r_squared,
           Adj_R_squared = adj_r_squared, F_statistic = f_stat,
           F_p_value = f_p_value, sig = sig_label(p_value)) %>%
    select(outcome, term, B, SE, CI_low, CI_high, t_value, p_value, sig,
           R_squared, Adj_R_squared, RMSE, F_statistic, F_p_value)
}

# --- Comparison Table (Before vs After) ---
build_comparison_table <- function(df_before, matched_df, targets_cols,
                                   binary_targets, output_dir) {
  comparison_table <- bind_rows(lapply(targets_cols, function(outcome) {
    form      <- as.formula(paste(outcome, "~ treated"))
    is_binary <- outcome %in% binary_targets
    fit_before  <- fit_model(form, df_before, is_binary)
    vcov_before <- vcovHC(fit_before, type = "HC3")
    test_before <- coeftest(fit_before, vcov = vcov_before)
    ci_before   <- coefci(fit_before, vcov. = vcov_before)
    fit_after  <- fit_model(form, matched_df, is_binary)
    vcov_after <- vcovCL(fit_after, cluster = matched_df$pair_id)
    test_after <- coeftest(fit_after, vcov = vcov_after)
    ci_after   <- coefci(fit_after, vcov. = vcov_after)
    build_row <- function(term_name, coef_name) tibble(
      outcome = outcome, term = term_name,
      B_before = test_before[coef_name, 1], SE_before = test_before[coef_name, 2],
      CI_low_before = ci_before[coef_name, 1], CI_high_before = ci_before[coef_name, 2],
      p_before = test_before[coef_name, 4],
      B_after = test_after[coef_name, 1], SE_after = test_after[coef_name, 2],
      CI_low_after = ci_after[coef_name, 1], CI_high_after = ci_after[coef_name, 2],
      p_after = test_after[coef_name, 4]
    )
    bind_rows(build_row("Treatment", "treated"), build_row("Intercept", "(Intercept)"))
  }))

  comparison_formatted <- comparison_table %>%
    mutate(N_before = nrow(df_before), N_after = nrow(matched_df),
           N_weighted = nrow(matched_df)) %>%
    mutate(across(c(B_before, B_after, SE_before, SE_after,
                    CI_low_before, CI_high_before, CI_low_after, CI_high_after),
                  ~ round(., 3))) %>%
    mutate(CI_before = paste0(CI_low_before, ", ", CI_high_before),
           CI_after  = paste0(CI_low_after, ", ", CI_high_after),
           B_before_fmt = paste0(B_before, sig_label(p_before)),
           B_after_fmt  = paste0(B_after, sig_label(p_after))) %>%
    select(outcome, term, B_before_fmt, SE_before, CI_before,
           B_after_fmt, SE_after, CI_after, N_before, N_after, N_weighted)
  write.csv(comparison_formatted, file.path(output_dir, "comparison_table.csv"),
            fileEncoding = "UTF-8", row.names = FALSE)
  comparison_formatted
}

# --- Love Plot ---
create_love_plot <- function(matchit_obj, output_dir) {
  library(cobalt); library(ggplot2)
  COL_BEFORE <- "#E64B35"; COL_AFTER <- "#3C5488"
  COL_GRID <- "#F0F0F0"; COL_TEXT <- "#000000"; FONT_FAMILY <- "sans"

  love_plot <- tryCatch(
    love.plot(matchit_obj, abs = TRUE, thresholds = 0.25, drop.distance = TRUE,
              var.order = "unadjusted",
              sample.names = c("Before Matching", "After Matching"),
              colors = c(COL_BEFORE, COL_AFTER), shapes = c(16, 16),
              stats = "mean.diffs", binary = "std", size = 3.5, line = FALSE),
    error = function(e) { message("love.plot failed: ", conditionMessage(e)); NULL }
  )
  if (is.null(love_plot)) return(invisible(NULL))

  polished_plot <- love_plot +
    labs(x = "Absolute Standardized Mean Difference", y = NULL,
         title = NULL, subtitle = NULL, colour = NULL, shape = NULL) +
    scale_y_discrete(labels = love_var_translate) +
    theme_minimal(base_size = 11, base_family = FONT_FAMILY) +
    theme(panel.grid.major.x = element_line(colour = COL_GRID, linewidth = 0.5),
          panel.grid.minor.x = element_blank(),
          panel.grid.major.y = element_line(colour = COL_GRID, linewidth = 0.5),
          axis.text.y = element_text(size = 10, colour = COL_TEXT, hjust = 1),
          axis.text.x = element_text(size = 10, colour = COL_TEXT),
          axis.title.x = element_text(size = 11, colour = COL_TEXT, margin = margin(t = 10)),
          legend.position = "bottom", legend.justification = "center",
          legend.box = "horizontal", legend.box.just = "center",
          legend.background = element_blank(), legend.key = element_blank(),
          legend.text = element_text(size = 10, colour = COL_TEXT),
          plot.margin = margin(12, 12, 12, 12)) +
    guides(colour = guide_legend(override.aes = list(size = 5.5)),
           shape = guide_legend(override.aes = list(size = 5.5)))

  num_vars <- length(unique(love_plot$data$var))
  calculated_height <- max(4, min(11, 2 + num_vars * 0.25))
  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
  ggsave(file.path(output_dir, "love_plot.png"), plot = polished_plot,
         width = 9.5, height = calculated_height, dpi = 300, bg = "white")
  invisible(love_plot)
}

In [ ]:
# =============================================================================
# SENSITIVITY KINDS (SM, Section 3.3)
#   impute / impute_0 / impute_6 / no_impute : data read from data/<kind>/
#   no_personal : primary imputed data, without age / marital status / criminal record
#   ps_overlap  : primary data and covariates again, to export full-sample propensity scores (ps_all.csv)
#   unfiltered  : includes cases without a declared sentencing range (incarceration only)
# =============================================================================
PERSONAL_COVARS <- c("age", "marital_status", "Criminal_record")

data_dir_for <- function(kind) if (kind %in% c("no_personal", "ps_overlap")) "impute" else kind

drop_personal_covars <- function(cv, kind) {
  if (kind != "no_personal") return(cv)
  lapply(cv, setdiff, PERSONAL_COVARS)
}

targets_for <- function(kind) {
  if (kind == "unfiltered") return("is_actual_sentence")
  c("min_punishment", "max_punishment", "is_actual_sentence")
}

# Analysis from csv

In [ ]:
analyze_pairs_from_csv <- function(
    df,
    pairs_path,
    output_dir,
    group_col,
    id_col,
    treatment,
    control,
    targets_cols,
    all_covars
) {
  
  # =========================
  # Helper Functions
  # =========================
  
  compute_cohens_d <- function(x_t, x_c) {
    n_t <- sum(!is.na(x_t))
    n_c <- sum(!is.na(x_c))
    if (n_t < 2 || n_c < 2) return(NA_real_)
    
    sd_pooled <- sqrt(((n_t - 1) * var(x_t, na.rm = TRUE) + 
                         (n_c - 1) * var(x_c, na.rm = TRUE)) / (n_t + n_c - 2))
    if (!is.finite(sd_pooled) || sd_pooled == 0) return(NA_real_)
    
    (mean(x_t, na.rm = TRUE) - mean(x_c, na.rm = TRUE)) / sd_pooled
  }
  
  smd_numeric <- function(x, t) {
    t_vals <- x[t == 1]
    c_vals <- x[t == 0]
    
    if (length(unique(na.omit(t_vals))) < 2 && length(unique(na.omit(c_vals))) < 2) {
      return(NA_real_)
    }
    
    vt <- var(t_vals, na.rm = TRUE)
    vc <- var(c_vals, na.rm = TRUE)
    
    if (!is.finite(vt) || !is.finite(vc) || (vt + vc) == 0) return(NA_real_)
    
    abs((mean(t_vals, na.rm = TRUE) - mean(c_vals, na.rm = TRUE)) / sqrt((vt + vc) / 2))
  }
  
  smd_factor <- function(x, t, varname) {
    levs <- levels(x)
    bind_rows(lapply(levs[-1], function(lv) {
      tibble(
        variable = paste0(varname, " = ", lv),
        smd = smd_numeric(as.numeric(x == lv), t)
      )
    }))
  }
  
  # =========================
  # Setup
  # =========================
  
  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
  
  # =========================
  # Covariates Setup
  # =========================
  
  # Exclude group column and related variables
  excluded <- group_col
  if (group_col %in% c("age_binary", "age_group")) {
    excluded <- c(excluded, "age")
  }
  if (group_col %in% c("location_binary", "location_group")) {
    excluded <- c(excluded, "location_group")
  }
  
  covars <- c(setdiff(all_covars, excluded))
  
  # =========================
  # Data Preprocessing
  # =========================
  
  df_before <- df %>%
    filter(.data[[group_col]] %in% c(treatment, control)) %>%
    drop_na(any_of(c(group_col, covars, targets_cols))) %>%
    mutate(
      treated = as.integer(.data[[group_col]] == treatment),
      verdict_id = as.character(.data[[id_col]])
    )
  
  df_group <- df_before
  
  # Log transform for continuous amount
  if ("total_amount" %in% names(df_group)) {
    df_group <- df_group %>%
      mutate(total_amount = log(total_amount + 0.001))
  }
  
  # Factorize categorical variables
  for (v in covars) {
    if (v %in% names(df_group)) {
      if (is.character(df_group[[v]]) || dplyr::n_distinct(df_group[[v]]) < 10) {
        df_group[[v]] <- as.factor(df_group[[v]])
      }
    }
  }


  # =========================
  # Reconstruct Matched Pairs
  # =========================
  
  pairs <- read_csv(pairs_path, show_col_types = FALSE) %>%
    mutate(
      pair_id = row_number(),
      across(c(id_treat, id_control), as.character)
    )
  
  pairs_long_map <- pairs %>%
    tidyr::pivot_longer(
      cols = c(id_treat, id_control),
      names_to = "type",
      values_to = "verdict_id"
    ) %>%
    mutate(treated = ifelse(type == "id_treat", 1L, 0L))
  
  matched_pairs_long <- df_group %>%
    inner_join(pairs_long_map, by = c("verdict_id", "treated")) %>%
    arrange(pair_id, desc(treated))

  # =========================
  # Calculate SMD (Before/After)
  # =========================
  
  smd_before <- bind_rows(lapply(covars, function(v) {
    x <- df_group[[v]]
    if (is.factor(x)) {
      smd_factor(x, df_group$treated, v) %>% mutate(time = "Before")
    } else {
      tibble(variable = v, smd = smd_numeric(x, df_group$treated), time = "Before")
    }
  }))
  
  smd_after <- bind_rows(lapply(covars, function(v) {
    x <- matched_pairs_long[[v]]
    if (is.factor(x)) {
      smd_factor(x, matched_pairs_long$treated, v) %>% mutate(time = "After")
    } else {
      tibble(variable = v, smd = smd_numeric(x, matched_pairs_long$treated), time = "After")
    }
  }))
  
  smd_long <- bind_rows(smd_before, smd_after) %>%
    filter(!is.na(smd))
  
  write_csv(smd_long, file.path(output_dir, "SMD.csv"))
  
  # =========================
  # Love Plot
  # =========================
  
  order_vars <- smd_long %>%
    filter(time == "Before") %>%
    arrange(smd) %>%
    pull(variable)
  
  smd_long$variable <- factor(smd_long$variable, levels = unique(order_vars))
  
  love_plot <- ggplot(smd_long, aes(x = smd, y = variable, color = time)) +
    geom_point(size = 2) +
    geom_vline(xintercept = 0.25, linetype = "dashed", alpha = 0.6) +
    scale_color_manual(values = c(Before = "#e11d48", After = "#0284c7")) +
    labs(title = "Covariate Balance", x = "|SMD|", y = NULL) +
    theme_minimal(base_size = 14) +
    theme(
      plot.background = element_rect(fill = "white", color = NA),
      panel.background = element_rect(fill = "white", color = NA),
      legend.background = element_rect(fill = "white", color = NA),
      legend.key = element_rect(fill = "white", color = NA)
    )
  
  ggsave(
    file.path(output_dir, "love_plot.png"),
    love_plot,
    width = 10,
    height = 7,
    dpi = 300,
    bg = "white"
  )
  
  # =========================
  # Outcome Analysis (Paired t-test)
  # =========================
  
  results_df <- bind_rows(lapply(targets_cols, function(outcome) {
    
    paired_data <- matched_pairs_long %>%
      select(pair_id, treated, !!sym(outcome)) %>%
      filter(is.finite(!!sym(outcome))) %>%
      group_by(pair_id) %>%
      filter(n() == 2) %>%
      arrange(pair_id, desc(treated)) %>%
      ungroup()
    
    if (nrow(paired_data) < 4) return(NULL)
    
    t_vals <- paired_data %>% filter(treated == 1) %>% pull(outcome)
    c_vals <- paired_data %>% filter(treated == 0) %>% pull(outcome)
    
    tt <- t.test(t_vals, c_vals, paired = TRUE)
    
    tibble(
      outcome        = outcome,
      mean_treatment = mean(t_vals),
      mean_control   = mean(c_vals),
      diff           = mean(t_vals) - mean(c_vals),
      std_error      = tt$stderr,
      ci_low         = tt$conf.int[1],
      ci_high        = tt$conf.int[2],
      d_cohens       = compute_cohens_d(t_vals, c_vals),
      p_value        = tt$p.value,
      n_pairs        = length(t_vals),
      n_treatment   = sum(!is.na(t_vals)),
      n_control     = sum(!is.na(c_vals)),
      sd_treatment  = sd(t_vals),
      sd_control    = sd(c_vals)
    )
  }))
  
  write_csv(results_df, file.path(output_dir, "results.csv"))
  
  # =========================
  # Comparison Table (Before vs After)
  # =========================
  
  comparison_table <- bind_rows(lapply(targets_cols, function(outcome) {
    
    form <- as.formula(paste(outcome, "~ treated"))
    
    # --- Before Matching (OLS) ---
    fit_before <- lm(form, data = df_before)
    coef_before <- summary(fit_before)$coefficients
    ci_before <- confint(fit_before)
    
    # --- After Matching (Paired data as OLS) ---
    paired_data <- matched_pairs_long %>%
      select(pair_id, treated, !!sym(outcome)) %>%
      filter(is.finite(!!sym(outcome))) %>%
      group_by(pair_id) %>%
      filter(n() == 2) %>%
      ungroup()
    
    if (nrow(paired_data) < 4) return(NULL)
    
    fit_after <- lm(form, data = paired_data)
    coef_after <- summary(fit_after)$coefficients
    ci_after <- confint(fit_after)
    
    treatment_row <- tibble(
      outcome = outcome,
      term = "Treatment",
      
      B_before = coef_before["treated", "Estimate"],
      SE_before = coef_before["treated", "Std. Error"],
      CI_low_before = ci_before["treated", 1],
      CI_high_before = ci_before["treated", 2],
      p_before = coef_before["treated", "Pr(>|t|)"],
      
      B_after = coef_after["treated", "Estimate"],
      SE_after = coef_after["treated", "Std. Error"],
      CI_low_after = ci_after["treated", 1],
      CI_high_after = ci_after["treated", 2],
      p_after = coef_after["treated", "Pr(>|t|)"]
    )
    
    intercept_row <- tibble(
      outcome = outcome,
      term = "Intercept",
      
      B_before = coef_before["(Intercept)", "Estimate"],
      SE_before = coef_before["(Intercept)", "Std. Error"],
      CI_low_before = ci_before["(Intercept)", 1],
      CI_high_before = ci_before["(Intercept)", 2],
      p_before = coef_before["(Intercept)", "Pr(>|t|)"],
      
      B_after = coef_after["(Intercept)", "Estimate"],
      SE_after = coef_after["(Intercept)", "Std. Error"],
      CI_low_after = ci_after["(Intercept)", 1],
      CI_high_after = ci_after["(Intercept)", 2],
      p_after = coef_after["(Intercept)", "Pr(>|t|)"]
    )
    
    bind_rows(treatment_row, intercept_row)
  }))
  
  if (!is.null(comparison_table) && nrow(comparison_table) > 0) {
    
    comparison_table <- comparison_table %>%
      mutate(
        N_before = nrow(df_before),
        N_after = nrow(matched_pairs_long),
        N_pairs = nrow(matched_pairs_long) / 2
      )
    
    comparison_formatted <- comparison_table %>%
      mutate(
        B_before = round(B_before, 3),
        B_after = round(B_after, 3),
        SE_before = round(SE_before, 3),
        SE_after = round(SE_after, 3),
        CI_low_before = round(CI_low_before, 3),
        CI_high_before = round(CI_high_before, 3),
        CI_low_after = round(CI_low_after, 3),
        CI_high_after = round(CI_high_after, 3),
        
        CI_before = paste0(CI_low_before, ", ", CI_high_before),
        CI_after = paste0(CI_low_after, ", ", CI_high_after),
        
        sig_before = case_when(
          p_before <= 0.001 ~ "***",
          p_before <= 0.01 ~ "**",
          p_before <= 0.05 ~ "*",
          TRUE ~ ""
        ),
        sig_after = case_when(
          p_after <= 0.001 ~ "***",
          p_after <= 0.01 ~ "**",
          p_after <= 0.05 ~ "*",
          TRUE ~ ""
        ),
        
        B_before_fmt = paste0(B_before, sig_before),
        B_after_fmt = paste0(B_after, sig_after)
      ) %>%
      select(
        outcome, term,
        B_before_fmt, SE_before, CI_before,
        B_after_fmt, SE_after, CI_after,
        N_before, N_after, N_pairs
      )
    
    write_csv(comparison_formatted, file.path(output_dir, "comparison_table.csv"))
    
  } else {
    comparison_formatted <- NULL
  }
  
  # =========================
  # Save Matched Data
  # =========================
  
  if ("total_amount" %in% names(matched_pairs_long)) {
    matched_pairs_long <- matched_pairs_long %>%
      mutate(total_amount = exp(total_amount) - 0.001)
  }
  
  write_csv(matched_pairs_long, file.path(output_dir, "matched.csv"))
  
  # =========================
  # Return
  # =========================
  # כמה יש לפני drop_na

  invisible(list(
    matched_pairs_long = matched_pairs_long,
    df_before          = df_before,
    smd_long           = smd_long,
    results            = results_df,
    comparison_table   = comparison_formatted,
    love_plot          = love_plot
  ))
}

# Run Defendants Race

In [ ]:
# =============================================================================
# CONFIG — WEAPONS, ETHNIC COMPARISON (Arab vs Jewish)
# =============================================================================
options(warn = -1)

CONFIG <- list(
  kinds       = c("no_impute","no_personal","unfiltered"),
  weapons     = c("carrying_only", "importing_only", "possession_only"),
  match_modes = c("logit"),
  group_col   = "ethnic",
  treatment   = "Arab",
  control     = "Jewish",
  caliper_q   = 0.9,
  caliper_ps  = 0.2,
  gower_demo_w = 1,
  gower_fact_w = 3,
  interaction_var = "war_period",
  war_start   = as.Date("2023-10-07"),
  near_constant_thresh = 0.85,
  use_punishment_months = FALSE
)

TARGETS <- list(
  main = c("min_punishment", "max_punishment", "is_actual_sentence"),
  pm   = c("punishment_months")
)
BINARY_TARGETS <- c("is_actual_sentence")

MULTILABEL_COLS <- c("weapon_motive", "weapon_use", "weapon_storage")

WEAPON_COUNT_CANDIDATES <- c(
  "handgun", "submachine_gun", "assault_rifle", "sniper_rifle", "hunting_rifle",
  "improvised_firearm", "grenade", "explosive_charge", "molotov",
  "anti_tank_missile", "ammunition_only", "weapon_parts", "unidentified_firearm"
)

# =============================================================================
# HELPER: expand a comma-separated multi-label column into 0/1 dummies
# =============================================================================
expand_multilabel_dummies <- function(df, col) {
  tags     <- lapply(strsplit(df[[col]], ","), stringr::str_squish)
  all_tags <- sort(unique(unlist(tags)))
  all_tags <- all_tags[!is.na(all_tags) & all_tags != ""]
  for (tag in all_tags)
    df[[paste0(col, "_", tag)]] <- as.integer(sapply(tags, function(x) tag %in% x))
  df[[col]] <- NULL
  df
}

# =============================================================================
# HELPER: covariates for a given weapon's data WITHOUT mutating globals
# =============================================================================
covars_for <- function(df) {
  # Keep only weapon-count columns that vary enough (one value < 85%)
  count_cols <- WEAPON_COUNT_CANDIDATES[sapply(WEAPON_COUNT_CANDIDATES, function(v) {
    if (!v %in% names(df)) return(FALSE)
    max(table(df[[v]])) / nrow(df) <= CONFIG$near_constant_thresh
  })]

  list(
    all = c("district","age","year","Court",
            count_cols,
            "actual_damage","premeditation",
            "escape_attempt","has_independent_offenses","undercover_sale",
            "magazines","bullets",
            "regret","confession","Criminal_record","marital_status",
            "ethnic","judge_ethnic"),
    fact = c("actual_damage","premeditation",
             "escape_attempt","has_independent_offenses","undercover_sale",
             "magazines","bullets",
             count_cols),
    demo = c("district","age","year","ethnic","regret","confession",
             "Court","Criminal_record","marital_status","judge_ethnic")
  )
}

# =============================================================================
# HELPER: load + clean + expand multi-labels for one weapon file
# =============================================================================
load_weapon_df <- function(kind, weapon) {
  df <- read_csv(paste0("data/", data_dir_for(kind), "/", weapon, ".csv"),
                 locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    mutate(
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      is_actual_sentence = as.integer(is_actual_sentence),
      Date               = dmy(Date),
      war_period         = if_else(Date >= CONFIG$war_start, 1L, 0L)
    ) %>%
    filter(ethnic %in% c("Arab", "Jewish")) %>%
    filter(!(judge_ethnic %in% c("Anonymous", "Unknown")))

  for (col in MULTILABEL_COLS)
    if (col %in% names(df)) df <- expand_multilabel_dummies(df, col)
  df
}

# =============================================================================
# HELPER: two-pass matching (main targets + optional punishment_months)
# =============================================================================
run_matching_two_passes <- function(df, output_dir, match_mode, cv,
                                     only_fact_vars = FALSE, targets = TARGETS$main) {
  prefix        <- if (only_fact_vars) "fact_match_" else "match_"
  method_subdir <- file.path(output_dir, paste0(prefix, match_mode))
  before_path   <- file.path(output_dir, "results_before_matching.csv")
  res_path      <- file.path(method_subdir, "results.csv")

  common_args <- list(
    group_col       = CONFIG$group_col,
    output_dir      = output_dir,
    treatment       = CONFIG$treatment,
    control         = CONFIG$control,
    match_mode      = match_mode,
    caliper_q       = CONFIG$caliper_q,
    caliper_ps_sd   = CONFIG$caliper_ps,
    all_covars      = cv$all,
    fact_covars     = cv$fact,
    demo_covars     = cv$demo,
    gower_demo_weight = CONFIG$gower_demo_w,
    gower_fact_weight = CONFIG$gower_fact_w,
    only_fact_vars  = only_fact_vars,
    interaction_var = CONFIG$interaction_var,
    war_interaction = TRUE
  )

  # --- Pass 1: main targets, full data ---
  cat("  [Pass 1] Main targets on full data\n")
  tryCatch(
    do.call(run_matching, c(list(df = df, targets_cols = targets,
                                  binary_targets = BINARY_TARGETS), common_args)),
    error = function(e) cat("  Error pass 1:", conditionMessage(e), "\n")
  )

  pass1_results <- if (file.exists(res_path))    read.csv(res_path,    fileEncoding = "UTF-8") else NULL
  pass1_before  <- if (file.exists(before_path)) read.csv(before_path, fileEncoding = "UTF-8") else NULL

  if (!CONFIG$use_punishment_months) return(invisible(NULL))

  # --- Pass 2: punishment_months, filtered data ---
  df_pm <- df %>% filter(punishment_months > 0)
  cat("  [Pass 2] punishment_months (N =", nrow(df_pm), ")\n")
  tryCatch(
    do.call(run_matching, c(list(df = df_pm, targets_cols = TARGETS$pm,
                                  binary_targets = NULL), common_args)),
    error = function(e) cat("  Error pass 2:", conditionMessage(e), "\n")
  )

  # --- Merge pass 1 + pass 2 ---
  if (!is.null(pass1_results) && file.exists(res_path)) {
    write.csv(rbind(pass1_results, read.csv(res_path, fileEncoding = "UTF-8")),
              res_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
  if (!is.null(pass1_before) && file.exists(before_path)) {
    write.csv(rbind(pass1_before, read.csv(before_path, fileEncoding = "UTF-8")),
              before_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
}

# =============================================================================
# MAIN LOOP
# =============================================================================
for (kind in CONFIG$kinds) {
  for (weapon in CONFIG$weapons) {
    df      <- load_weapon_df(kind, weapon)
    cv      <- drop_personal_covars(covars_for(df), kind)   # local copy — no global mutation
    out_dir <- paste0("matched_outputs/", kind, "/", weapon, "/Defendants_Ethnic")

    cat("\n=== Processing:", kind, "/", weapon, "===\n")

    for (mm in CONFIG$match_modes) {
      cat("\n--- match_mode:", mm, "---\n")
      run_matching_two_passes(df = df, output_dir = out_dir,
                              match_mode = mm, cv = cv, only_fact_vars = FALSE,
                              targets = targets_for(kind))
    }
  }
}

# # --- Pair-based analysis ---
# cat("\n--- Analyzing pairs from CSV ---\n")
# tryCatch(
#   analyze_pairs_from_csv(
#     df           = df,
#     pairs_path   = paste0("data/pairs/", CONFIG$group_col, "/", weapon, ".csv"),
#     output_dir   = file.path(out_dir, "manual_pairs"),
#     group_col    = CONFIG$group_col,
#     id_col       = "original_case",
#     treatment    = CONFIG$treatment,
#     control      = CONFIG$control,
#     targets_cols = TARGETS$main,
#     all_covars   = cv$all
#   ),
#   error = function(e) cat("  Error in pair analysis:", conditionMessage(e), "\n")
#     )

# District

In [ ]:
# =============================================================================
# WEAPONS — DISTRICT ANALYSIS, ONE VS ALL COMPARISONS
# =============================================================================
options(warn = -1)

CONFIG <- list(
  kinds       = c("no_impute","no_personal"),
  # weapons       = c("possession_only"),
  weapons       = c("carrying_only", "importing_only", "possession_only"),
  match_modes   = c("logit"),
  caliper_q     = 0.9,
  caliper_ps    = 0.2,
  gower_demo_w  = 1,
  gower_fact_w  = 3,
  min_district_n = 10,
  min_group_n    = 10,
  control_label  = "All_Others",
  group_col      = "district_one_vs_all",
  near_constant_thresh = 0.95,
  bonferroni_k   = 30,            # 6 districts × 5 comparisons (drugs & weapons), pre-specified
  use_punishment_months = FALSE
)

TARGETS <- list(
  main = c("min_punishment", "max_punishment", "is_actual_sentence"),
  pm   = c("punishment_months")
)
BINARY_TARGETS <- c("is_actual_sentence")

MULTILABEL_COLS <- c("weapon_motive", "weapon_use", "weapon_storage")

WEAPON_COUNT_CANDIDATES <- c(
  "handgun", "submachine_gun", "assault_rifle", "sniper_rifle", "hunting_rifle",
  "improvised_firearm", "grenade", "explosive_charge", "molotov",
  "anti_tank_missile", "ammunition_only", "weapon_parts", "unidentified_firearm"
)

# =============================================================================
# HELPER: expand a comma-separated multi-label column into 0/1 dummies
# =============================================================================
expand_multilabel_dummies <- function(df, col) {
  tags     <- lapply(strsplit(df[[col]], ","), stringr::str_squish)
  all_tags <- sort(unique(unlist(tags)))
  all_tags <- all_tags[!is.na(all_tags) & all_tags != ""]
  for (tag in all_tags)
    df[[paste0(col, "_", tag)]] <- as.integer(sapply(tags, function(x) tag %in% x))
  df[[col]] <- NULL
  df
}

# =============================================================================
# HELPER: load + clean + expand multi-labels for one weapon file
# =============================================================================
load_weapon_df <- function(kind, weapon) {
  df <- read_csv(paste0("data/", data_dir_for(kind), "/", weapon, ".csv"),
                 locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    filter(Criminal_record != "לא ידוע") %>%
    mutate(
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      is_actual_sentence = as.integer(is_actual_sentence)
    ) %>%
    filter(ethnic %in% c("Arab", "Jewish")) %>%
    filter(!(judge_ethnic %in% c("Anonymous", "Unknown")))

  for (col in MULTILABEL_COLS)
    if (col %in% names(df)) df <- expand_multilabel_dummies(df, col)
  df
}

# =============================================================================
# HELPER: covariates for this weapon's data (district EXCLUDED — it's the group)
# =============================================================================
covars_for <- function(df) {
  count_cols <- WEAPON_COUNT_CANDIDATES[sapply(WEAPON_COUNT_CANDIDATES, function(v) {
    if (!v %in% names(df)) return(FALSE)
    max(table(df[[v]])) / nrow(df) <= CONFIG$near_constant_thresh
  })]

  list(
    all = c("age","marital_status","year","Court",
            count_cols,
            "actual_damage","premeditation",
            "escape_attempt","has_independent_offenses","undercover_sale",
            "magazines","bullets",
            "ethnic","regret","confession","Criminal_record","judge_ethnic"),
    fact = c("actual_damage","premeditation",
             "escape_attempt","has_independent_offenses","undercover_sale",
             "magazines","bullets",
             count_cols),
    demo = c("age","marital_status","year","ethnic",
             "regret","confession","Court","Criminal_record","judge_ethnic")
  )
}

# =============================================================================
# HELPER: districts with enough observations
# =============================================================================
eligible_districts <- function(df) {
  df %>%
    filter(!is.na(district)) %>%
    count(district, name = "n") %>%
    filter(n >= CONFIG$min_district_n) %>%
    arrange(district) %>%
    pull(district)
}

# =============================================================================
# HELPER: two-pass matching (main + optional punishment_months)
# =============================================================================
run_matching_two_passes <- function(df, output_dir, treatment, control,
                                     match_mode, cv, only_fact_vars = FALSE) {
  prefix        <- if (only_fact_vars) "fact_match_" else "match_"
  method_subdir <- file.path(output_dir, paste0(prefix, match_mode))
  before_path   <- file.path(output_dir, "results_before_matching.csv")
  res_path      <- file.path(method_subdir, "results.csv")

  common_args <- list(
    group_col       = CONFIG$group_col,
    output_dir      = output_dir,
    treatment       = treatment,
    control         = control,
    match_mode      = match_mode,
    caliper_q       = CONFIG$caliper_q,
    caliper_ps_sd   = CONFIG$caliper_ps,
    all_covars      = cv$all,
    fact_covars     = cv$fact,
    demo_covars     = cv$demo,
    gower_demo_weight = CONFIG$gower_demo_w,
    gower_fact_weight = CONFIG$gower_fact_w,
    only_fact_vars  = only_fact_vars,
    bonferroni_k    = CONFIG$bonferroni_k     # 30 for districts
  )

  cat("   [Pass 1] Main targets on full data\n")
  tryCatch(
    do.call(run_matching, c(list(df = df, targets_cols = TARGETS$main,
                                  binary_targets = BINARY_TARGETS), common_args)),
    error = function(e) cat("   Error pass 1:", conditionMessage(e), "\n")
  )

  pass1_results <- if (file.exists(res_path))    read.csv(res_path,    fileEncoding = "UTF-8") else NULL
  pass1_before  <- if (file.exists(before_path)) read.csv(before_path, fileEncoding = "UTF-8") else NULL

  if (!CONFIG$use_punishment_months) return(invisible(NULL))

  df_pm <- df %>% filter(punishment_months > 0)
  cat("   [Pass 2] punishment_months (N =", nrow(df_pm), ")\n")
  tryCatch(
    do.call(run_matching, c(list(df = df_pm, targets_cols = TARGETS$pm,
                                  binary_targets = NULL), common_args)),
    error = function(e) cat("   Error pass 2:", conditionMessage(e), "\n")
  )

  if (!is.null(pass1_results) && file.exists(res_path)) {
    write.csv(rbind(pass1_results, read.csv(res_path, fileEncoding = "UTF-8")),
              res_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
  if (!is.null(pass1_before) && file.exists(before_path)) {
    write.csv(rbind(pass1_before, read.csv(before_path, fileEncoding = "UTF-8")),
              before_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
}

# =============================================================================
# HELPER: process one district (one-vs-all)
# =============================================================================
run_one_district <- function(df, weapon, target_district, cv, idx, total) {
  folder_name <- paste0("district_", gsub(" ", "_", target_district), "_vs_All_Others")
  out_dir     <- paste0("matched_outputs/", weapon, "/", folder_name)

  cat("\n========================================\n")
  cat(sprintf("DISTRICT %d/%d: %s vs All Others\n", idx, total, target_district))
  cat("========================================\n")

  df_temp <- df %>%
    mutate(district_one_vs_all = if_else(district == target_district,
                                         target_district, CONFIG$control_label))

  n_t <- sum(df_temp$district_one_vs_all == target_district,      na.rm = TRUE)
  n_c <- sum(df_temp$district_one_vs_all == CONFIG$control_label, na.rm = TRUE)
  cat(sprintf("Sample sizes: %s = %d, %s = %d\n",
              target_district, n_t, CONFIG$control_label, n_c))

  if (n_t < CONFIG$min_group_n || n_c < CONFIG$min_group_n) {
    cat("WARNING: Skipping due to small sample size\n")
    return(invisible(NULL))
  }

  for (mm in CONFIG$match_modes) {
    cat("\n--- match_mode:", mm, "---\n")
    tryCatch(
      run_matching_two_passes(df = df_temp, output_dir = out_dir,
                              treatment = target_district, control = CONFIG$control_label,
                              match_mode = mm, cv = cv, only_fact_vars = FALSE),
      error = function(e) cat("Error in matching for", mm, ":", conditionMessage(e), "\n")
    )
  }
}

# =============================================================================
# HELPER: collect one-vs-all summary for a weapon
# =============================================================================
collect_summary <- function(weapon, districts) {
  cat("\n========================================\n")
  cat("COLLECTING ONE-VS-ALL SUMMARY RESULTS\n")
  cat("========================================\n")

  summary_results <- lapply(districts, function(target_district) {
    folder_name  <- paste0("district_", gsub(" ", "_", target_district), "_vs_All_Others")
    results_path <- paste0("matched_outputs/", weapon, "/", folder_name, "/match_logit/results.csv")
    if (!file.exists(results_path)) return(NULL)
    tryCatch(
      read_csv(results_path, show_col_types = FALSE) %>%
        mutate(comparison = paste0(target_district, " vs All Others"),
               treatment  = target_district,
               control    = CONFIG$control_label),
      error = function(e) { cat("Could not read results for", folder_name, "\n"); NULL }
    )
  })

  summary_results <- Filter(Negate(is.null), summary_results)
  if (length(summary_results) == 0) return(invisible(NULL))

  all_results <- bind_rows(summary_results)
  summary_path <- paste0("matched_outputs/", weapon, "/district_one_vs_all_summary.csv")
  write_csv(all_results, summary_path)
  cat("\nSummary saved to:", summary_path, "\n")

  # p_bonferroni already computed upstream (bonferroni_k = 30). Stars use corrected p.
  summary_table <- all_results %>%
    select(comparison, outcome, diff, std_error, p_value, p_bonferroni, d_cohens) %>%
    mutate(
      sig = case_when(
        p_bonferroni <= 0.001 ~ "***",
        p_bonferroni <= 0.01  ~ "**",
        p_bonferroni <= 0.05  ~ "*",
        p_bonferroni <= 0.1   ~ ".",
        TRUE                  ~ ""
      ),
      diff_fmt = sprintf("%.2f%s", diff, sig)
    )

  cat("\n=== SUMMARY: Treatment Effects by District (One vs All, Bonferroni-corrected) ===\n\n")
  print(summary_table, n = 50)
  invisible(all_results)
}

# =============================================================================
# MAIN LOOP
# =============================================================================
for (kind in CONFIG$kinds) {
  for (weapon_name in CONFIG$weapons) {
    weapon <- paste0(kind, "/", weapon_name)
    cat("\n=== Processing:", weapon, "===\n")

    df <- load_weapon_df(kind, weapon_name)
    cv <- drop_personal_covars(covars_for(df), kind)

    cat("\n=== District Distribution (", weapon_name, ") ===\n")
    print(table(df$district))

    districts <- eligible_districts(df)
    cat("\nTotal One-vs-All comparisons:", length(districts), "\n")
    for (i in seq_along(districts))
      cat(sprintf("   %2d. %s vs All Others\n", i, districts[i]))

    for (i in seq_along(districts))
      run_one_district(df, weapon, districts[i], cv, i, length(districts))

    collect_summary(weapon, districts)

    cat("\n========================================\n")
    cat("Done:", weapon, "\n")
    cat("========================================\n")
  }
}

## one vs one

In [ ]:
# =============================================================================
# DISTRICT ANALYSIS - ALL PAIRWISE COMPARISONS
# =============================================================================

run_matching_two_passes <- function(df, group_col, output_dir, treatment, control,
                                     match_mode, only_fact_vars = FALSE, ...) {
  prefix <- if (only_fact_vars) "fact_match_" else "match_"
  method_subdir <- file.path(output_dir, paste0(prefix, match_mode))
  before_path <- file.path(output_dir, "results_before_matching.csv")

  cat("  [Pass 1] Main targets on full data\n")
  tryCatch({
    run_matching(df = df, group_col = group_col, targets_cols = targets_main,
                 output_dir = output_dir, treatment = treatment, control = control,
                 match_mode = match_mode, only_fact_vars = only_fact_vars,
                 binary_targets = c("is_actual_sentence"), ...)
  }, error = function(e) cat("  Error pass 1:", conditionMessage(e), "\n"))

  pass1_results <- NULL
  res_path <- file.path(method_subdir, "results.csv")
  if (file.exists(res_path)) pass1_results <- read.csv(res_path, fileEncoding = "UTF-8")
  pass1_before <- NULL
  if (file.exists(before_path)) pass1_before <- read.csv(before_path, fileEncoding = "UTF-8")

  if (!use_punishment_months) return(invisible(NULL))

  df_pm <- df %>% filter(punishment_months > 0)
  cat("  [Pass 2] punishment_months on filtered data (N=", nrow(df_pm), ")\n")
  tryCatch({
    run_matching(df = df_pm, group_col = group_col, targets_cols = targets_pm,
                 output_dir = output_dir, treatment = treatment, control = control,
                 match_mode = match_mode, only_fact_vars = only_fact_vars,
                 binary_targets = NULL, ...)
  }, error = function(e) cat("  Error pass 2:", conditionMessage(e), "\n"))

  if (!is.null(pass1_results) && file.exists(res_path)) {
    merged <- rbind(pass1_results, read.csv(res_path, fileEncoding = "UTF-8"))
    write.csv(merged, res_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
  if (!is.null(pass1_before) && file.exists(before_path)) {
    merged_before <- rbind(pass1_before, read.csv(before_path, fileEncoding = "UTF-8"))
    write.csv(merged_before, before_path, fileEncoding = "UTF-8", row.names = FALSE)
  }
}

targets_main <- c("min_punishment", "max_punishment", "is_actual_sentence")
targets_pm   <- c("punishment_months")
use_punishment_months <- FALSE

expand_multilabel_dummies <- function(df, col) {
  tags     <- lapply(stringr::str_squish(strsplit(df[[col]], ",")), stringr::str_squish)
  all_tags <- sort(unique(unlist(tags)))
  all_tags <- all_tags[!is.na(all_tags) & all_tags != ""]
  for (tag in all_tags)
    df[[paste0(col, "_", tag)]] <- as.integer(sapply(tags, function(x) tag %in% x))
  df[[col]] <- NULL
  df
}

weapons <- c("carrying_only", "importing_only", "possession_only")
kind    <- c("impute")

for (kind_val in kind) {
  for (w in weapons) {
    weapon <- paste0(kind_val, "/", w)
    print(paste0("Processing weapon: ", weapon))

    # ── Load & clean ──────────────────────────────────────────────────────────
    df <- read_csv(paste0("data/", weapon, ".csv"),
                   locale = locale(encoding = "UTF-8"), show_col_types = FALSE)

    df <- df %>%
      filter(Criminal_record != "לא ידוע") %>%
      mutate(
        district           = str_squish(as.character(district)),
        age                = as.numeric(age),
        is_actual_sentence = as.integer(is_actual_sentence)
      ) %>%
      filter(ethnic %in% c("Arab", "Jewish")) %>%
      filter(!(judge_ethnic %in% c("Anonymous", "Unknown")))

    # ── Multi-label expansion ─────────────────────────────────────────────────
    multilabel_cols <- c("weapon_motive", "weapon_use", "weapon_storage")
    for (col in multilabel_cols) {
      if (col %in% names(df)) df <- expand_multilabel_dummies(df, col)
    }

    # ── Dynamic covars ────────────────────────────────────────────────────────
    numeric_weapon_cols <- c(
      "handgun", "submachine_gun", "assault_rifle", "sniper_rifle", "hunting_rifle",
      "improvised_firearm", "grenade", "explosive_charge", "molotov", "anti_tank_missile",
      "ammunition_only", "weapon_parts", "unidentified_firearm"
    )
    numeric_weapon_cols <- numeric_weapon_cols[sapply(numeric_weapon_cols, function(col) {
      if (!col %in% names(df)) return(FALSE)
      max(table(df[[col]])) / nrow(df) <= 0.95
    })]

    dummy_cols <- names(df)[
      Reduce(`|`, lapply(multilabel_cols, function(col) startsWith(names(df), paste0(col, "_"))))
    ]

    all_covars_district <- c(
      "age", "marital_status", "year", "Court",
      numeric_weapon_cols, 
      # dummy_cols,
      "actual_damage", "premeditation",
      "escape_attempt", "has_independent_offenses", "undercover_sale",
      "magazines", "bullets",
      "ethnic", "regret", "confession", "Criminal_record", "judge_ethnic"
    )

    fact_covars <- c(
      dummy_cols,
      "actual_damage", "potential_damage", "premeditation",
      "escape_attempt", "has_independent_offenses", "undercover_sale",
      "magazines", "bullets", "judge_ethnic"
    )

    demo_covars <- c(
      "age", "marital_status", "year", "ethnic",
      "regret", "confession", "Court", "Criminal_record"
    )

    exact_vars <- NULL

    # ── Districts with sufficient N ───────────────────────────────────────────
    cat("\n=== District Distribution (", w, ") ===\n")
    print(table(df$district))

    districts <- df %>%
      filter(!is.na(district)) %>%
      count(district, name = "n") %>%
      filter(n >= 0) %>%
      arrange(district) %>%
      pull(district)

    district_pairs <- tryCatch({
      if (length(districts) < 2) {
        cat("WARNING: Not enough districts (found", length(districts), ")\n")
        list()
      } else {
        combn(districts, 2, simplify = FALSE)
      }
    }, error = function(e) { cat("ERROR:", e$message, "\n"); list() })

    cat("\nTotal pairwise comparisons:", length(district_pairs), "\n")
    for (i in seq_along(district_pairs))
      cat(sprintf("  %2d. %s vs %s\n", i, district_pairs[[i]][1], district_pairs[[i]][2]))

    # ── Run matching for all pairs ────────────────────────────────────────────
    for (pair_idx in seq_along(district_pairs)) {
      dist1 <- district_pairs[[pair_idx]][1]
      dist2 <- district_pairs[[pair_idx]][2]
      folder_name <- paste0("district_", gsub(" ", "_", dist1), "_vs_", gsub(" ", "_", dist2))

      cat("\n========================================\n")
      cat(sprintf("PAIR %d/%d: %s vs %s\n", pair_idx, length(district_pairs), dist1, dist2))
      cat("========================================\n")

      n1 <- sum(df$district == dist1, na.rm = TRUE)
      n2 <- sum(df$district == dist2, na.rm = TRUE)
      cat(sprintf("Sample sizes: %s = %d, %s = %d\n", dist1, n1, dist2, n2))

      if (n1 < 10 || n2 < 10) { cat("WARNING: Skipping (< 10)\n"); next }

      tryCatch({
        run_matching_two_passes(
          df = df, group_col = "district",
          output_dir = paste0("matched_outputs/", weapon, "/", folder_name),
          treatment = dist1, control = dist2,
          caliper_q = 0.2, caliper_ps_sd = 0.2, match_mode = "logit",
          exact_vars = exact_vars,
          all_covars = all_covars_district,
          fact_covars = fact_covars,
          demo_covars = demo_covars,
          gower_demo_weight = 1, gower_fact_weight = 3,
          only_fact_vars = FALSE
        )
      }, error = function(e) cat("Error:", e$message, "\n"))
    }

    # ── Summary table ─────────────────────────────────────────────────────────
    cat("\n========================================\n")
    cat("COLLECTING SUMMARY RESULTS\n")
    cat("========================================\n")

    summary_results <- list()
    for (pair_idx in seq_along(district_pairs)) {
      dist1 <- district_pairs[[pair_idx]][1]
      dist2 <- district_pairs[[pair_idx]][2]
      folder_name  <- paste0("district_", gsub(" ", "_", dist1), "_vs_", gsub(" ", "_", dist2))
      results_path <- paste0("matched_outputs/", weapon, "/", folder_name, "/match_logit/results.csv")

      if (file.exists(results_path)) {
        tryCatch({
          res <- read_csv(results_path, show_col_types = FALSE) %>%
            mutate(comparison = paste0(dist1, " vs ", dist2),
                   treatment = dist1, control = dist2)
          summary_results[[pair_idx]] <- res
        }, error = function(e) cat("Could not read results for", folder_name, "\n"))
      }
    }

    if (length(summary_results) > 0) {
      all_results <- bind_rows(summary_results)
      write_csv(all_results, paste0("matched_outputs/", weapon, "/district_all_pairs_summary.csv"))
      cat("\nSummary saved.\n")

      all_results %>%
        select(comparison, outcome, diff, std_error, p_value, d_cohens) %>%
        mutate(
          sig      = case_when(p_value <= 0.001 ~ "***", p_value <= 0.01 ~ "**",
                               p_value <= 0.05 ~ "*",   p_value <= 0.1  ~ ".", TRUE ~ ""),
          diff_fmt = sprintf("%.2f%s", diff, sig)
        ) %>%
        print(n = 50)
    }

    cat("\n========================================\n")
    cat("Done:", weapon, "\n")
    cat("========================================\n")
  }
}


# year

In [ ]:
# =============================================================================
# WEAPONS — TIME-PERIOD ANALYSIS (Corona / War)
# =============================================================================

CONFIG <- list(
  kinds       = c("no_impute","no_personal"),
  weapons     = c("carrying_only", "importing_only", "possession_only"),
  match_modes = c("logit"),
  min_n       = 10,
  caliper_q   = 0.9,
  caliper_ps  = 0.2,
  gower_demo_w = 1,
  gower_fact_w = 3,
  war_start   = as.Date("2023-10-07"),
  corona_start = as.Date("2020-03-01"),
  near_constant_thresh = 0.95,        # <<< אותו ערך כמו שאר ה-drivers — תאשר
  use_punishment_months = FALSE
)

TARGETS        <- c("min_punishment", "max_punishment", "is_actual_sentence")
BINARY_TARGETS <- c("is_actual_sentence")

PERIOD_LABELS <- list(
  corona_period = list(control = "pre_corona", treatment = "post_corona"),
  war_period    = list(control = "pre_war",    treatment = "post_war")
)

MULTILABEL_COLS <- c("weapon_motive", "weapon_use", "weapon_storage")

WEAPON_COUNT_CANDIDATES <- c(
  "handgun", "submachine_gun", "assault_rifle", "sniper_rifle", "hunting_rifle",
  "improvised_firearm", "grenade", "explosive_charge", "molotov",
  "anti_tank_missile", "ammunition_only", "weapon_parts", "unidentified_firearm"
)

# =============================================================================
# HELPER: expand a comma-separated multi-label column into 0/1 dummies
# =============================================================================
expand_multilabel_dummies <- function(df, col) {
  tags     <- lapply(strsplit(df[[col]], ","), stringr::str_squish)
  all_tags <- sort(unique(unlist(tags)))
  all_tags <- all_tags[!is.na(all_tags) & all_tags != ""]
  for (tag in all_tags)
    df[[paste0(col, "_", tag)]] <- as.integer(sapply(tags, function(x) tag %in% x))
  df[[col]] <- NULL
  df
}

# =============================================================================
# HELPER: covariate sets for this weapon's data
# =============================================================================
covars_for <- function(df) {
  count_cols <- WEAPON_COUNT_CANDIDATES[sapply(WEAPON_COUNT_CANDIDATES, function(v) {
    if (!v %in% names(df)) return(FALSE)
    max(table(df[[v]])) / nrow(df) <= CONFIG$near_constant_thresh
  })]

  list(
    all = c("district","marital_status","Court","age",
            count_cols,
            "actual_damage","premeditation",
            "escape_attempt","has_independent_offenses","undercover_sale",
            "magazines","bullets",
            "ethnic","regret","confession","Criminal_record","judge_ethnic"),
    fact = c("actual_damage","premeditation",
             "escape_attempt","has_independent_offenses","undercover_sale",
             "magazines","bullets",
             count_cols),
    demo = c("district","age","marital_status","ethnic",
             "regret","confession","Court","Criminal_record","judge_ethnic")
  )
}

# =============================================================================
# HELPER: load + clean + expand multi-labels for one weapon file
# =============================================================================
load_weapon_df <- function(kind, weapon) {
  df <- read_csv(paste0("data/", data_dir_for(kind), "/", weapon, ".csv"),
                 locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    filter(Criminal_record != "לא ידוע") %>%
    mutate(
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      is_actual_sentence = as.integer(is_actual_sentence),
      Date               = dmy(Date),
      war_period         = if_else(Date >= CONFIG$war_start, 1L, 0L),
      corona_period      = if_else(Date >= CONFIG$corona_start, 1L, 0L)
    ) %>%
    filter(ethnic %in% c("Arab", "Jewish")) %>%
    filter(!(judge_ethnic %in% c("Anonymous", "Unknown")))

  for (col in MULTILABEL_COLS)
    if (col %in% names(df)) df <- expand_multilabel_dummies(df, col)
  df
}

# =============================================================================
# HELPER: one time-period comparison
# =============================================================================
run_time_comparison <- function(df, period_col, comparison_name, weapon, kind,
                                 binary_targets = BINARY_TARGETS) {

  if (!period_col %in% names(df)) {
    cat("WARNING: column '", period_col, "' not found. Skipping.\n")
    return(invisible(NULL))
  }
  if (!period_col %in% names(PERIOD_LABELS)) stop("Unknown period_col: ", period_col)

  lbl       <- PERIOD_LABELS[[period_col]]
  group_col <- paste0(period_col, "_group")

  df_local <- df %>%
    mutate(!!group_col := case_when(
      .data[[period_col]] == 1 ~ lbl$treatment,
      .data[[period_col]] == 0 ~ lbl$control,
      TRUE                     ~ NA_character_
    ))

  cat("\n========================================\n")
  cat("TIME COMPARISON:", comparison_name, "| period:", period_col, "\n")
  cat("========================================\n")
  print(table(df_local[[group_col]], useNA = "ifany"))

  n_treat <- sum(df_local[[group_col]] == lbl$treatment, na.rm = TRUE)
  n_ctrl  <- sum(df_local[[group_col]] == lbl$control,   na.rm = TRUE)
  cat(sprintf("\nSample sizes: %s = %d, %s = %d\n",
              lbl$treatment, n_treat, lbl$control, n_ctrl))

  if (n_treat < CONFIG$min_n || n_ctrl < CONFIG$min_n) {
    cat("\nWARNING: Skipping — insufficient sample (min:", CONFIG$min_n, ")\n")
    return(invisible(NULL))
  }

  cv      <- drop_personal_covars(covars_for(df), kind)
  out_dir <- file.path("matched_outputs", weapon, comparison_name)

  # period IS the treatment → war_interaction stays FALSE (collinearity guard)
  interaction_var <- if (period_col == "war_period") "war_period" else "ethnic"

  for (mm in CONFIG$match_modes) {
    cat("\n--- match_mode:", mm, "---\n")
    tryCatch(
      run_matching(
        df = df_local, group_col = group_col, targets_cols = TARGETS,
        output_dir = out_dir, treatment = lbl$treatment, control = lbl$control,
        caliper_q = CONFIG$caliper_q, caliper_ps_sd = CONFIG$caliper_ps,
        match_mode = mm,
        all_covars = cv$all, fact_covars = cv$fact, demo_covars = cv$demo,
        gower_demo_weight = CONFIG$gower_demo_w, gower_fact_weight = CONFIG$gower_fact_w,
        only_fact_vars = FALSE, binary_targets = binary_targets,
        interaction_var = interaction_var
      ),
      error = function(e) cat("Error in matching for", mm, ":", conditionMessage(e), "\n")
    )
  }
  invisible(TRUE)
}

# =============================================================================
# MAIN LOOP
# =============================================================================
for (kind in CONFIG$kinds) {
  for (weapon_name in CONFIG$weapons) {
    weapon <- paste0(kind, "/", weapon_name)
    cat("\n=== Processing:", weapon, "===\n")
    df <- load_weapon_df(kind, weapon_name)

    for (time_col in c("corona_period", "war_period")) {
      comparison_name <- gsub("_period$", "", time_col)
      cat("\n--- Comparison:", comparison_name, "for", weapon, "---\n")
      tryCatch(
        run_time_comparison(df, time_col, comparison_name, weapon, kind),
        error = function(e) cat("ERROR in", time_col, ":", conditionMessage(e), "\n")
      )
    }
    cat("\n=== Time comparisons complete for", weapon, "===\n")
  }
}

In [ ]:
# =============================================================================
# run_ethnic_by_war_period_stratified_weapons.R
# PSM stratified by sector (Arab / Jewish): pre_war vs. war, per weapon.
# Q: Did sentencing change after the war — separately within each sector?
# =============================================================================

CONFIG <- list(
  kinds       = c("no_impute","no_personal"),
  weapons     = c("carrying_only", "possession_only"),   # importing too small
  sectors     = c("Arab", "Jewish"),
  war_cutoff  = as.Date("2023-10-07"),
  min_n       = 10,
  caliper_ps  = 0.2,
  match_mode  = "logit",
  near_constant_thresh = 0.85          # <<< keep consistent across all weapon drivers
)

TARGETS        <- c("min_punishment", "max_punishment", "is_actual_sentence")
BINARY_TARGETS <- c("is_actual_sentence")

MULTILABEL_COLS <- c("weapon_motive", "weapon_use", "weapon_storage")

WEAPON_COUNT_CANDIDATES <- c(
  "handgun", "submachine_gun", "assault_rifle", "sniper_rifle", "hunting_rifle",
  "improvised_firearm", "grenade", "explosive_charge", "molotov",
  "anti_tank_missile", "ammunition_only", "weapon_parts", "unidentified_firearm"
)

# =============================================================================
# HELPER: expand a comma-separated multi-label column into 0/1 dummies
# =============================================================================
expand_multilabel_dummies <- function(df, col) {
  tags     <- lapply(strsplit(df[[col]], ","), stringr::str_squish)
  all_tags <- sort(unique(unlist(tags)))
  all_tags <- all_tags[!is.na(all_tags) & all_tags != ""]
  for (tag in all_tags)
    df[[paste0(col, "_", tag)]] <- as.integer(sapply(tags, function(x) tag %in% x))
  df[[col]] <- NULL
  df
}

# =============================================================================
# HELPER: covariate sets (ethnic excluded — sector is fixed within each loop)
# =============================================================================
covars_for <- function(df) {
  count_cols <- WEAPON_COUNT_CANDIDATES[sapply(WEAPON_COUNT_CANDIDATES, function(v) {
    if (!v %in% names(df)) return(FALSE)
    max(table(df[[v]])) / nrow(df) <= CONFIG$near_constant_thresh
  })]

  list(
    all = c("district","age","Court",
            count_cols,
            "actual_damage","premeditation",
            "escape_attempt","has_independent_offenses","undercover_sale",
            "magazines","bullets",
            "regret","confession","Criminal_record","marital_status","judge_ethnic"),
    fact = c(count_cols,
             "actual_damage","premeditation",
             "escape_attempt","has_independent_offenses","undercover_sale",
             "magazines","bullets"),
    demo = c("district","age","Court",
             "regret","confession","Criminal_record","marital_status","judge_ethnic")
  )
}

# =============================================================================
# HELPER: load + clean + expand multi-labels for one weapon file
# =============================================================================
load_weapon_df <- function(kind, weapon) {
  df <- read_csv(paste0("data/", data_dir_for(kind), "/", weapon, ".csv"),
                 locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    mutate(
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      year               = as.numeric(as.character(year)),
      is_actual_sentence = as.integer(is_actual_sentence),
      Date               = dmy(Date),
      war_period         = if_else(Date >= CONFIG$war_cutoff, "war", "pre_war")
    ) %>%
    filter(ethnic %in% c("Arab", "Jewish")) %>%
    filter(!(judge_ethnic %in% c("Anonymous", "Unknown"))) %>%
    filter(Criminal_record != "לא ידוע")

  for (col in MULTILABEL_COLS)
    if (col %in% names(df)) df <- expand_multilabel_dummies(df, col)
  df
}

# =============================================================================
# HELPER: drop covariates that are constant within a subgroup
# =============================================================================
keep_varying_covars <- function(covars, df_sub) {
  covars[sapply(covars, function(v) {
    if (!v %in% names(df_sub)) return(FALSE)
    length(unique(na.omit(as.character(df_sub[[v]])))) >= 2
  })]
}

# =============================================================================
# HELPER: run one sector × weapon matching
# =============================================================================
run_sector <- function(df_sector, weapon, sector, cv, kind) {
  cat("\n  Sector:", sector, "\n")
  print(df_sector %>% count(war_period))

  n_pre <- sum(df_sector$war_period == "pre_war", na.rm = TRUE)
  n_war <- sum(df_sector$war_period == "war",     na.rm = TRUE)
  if (n_pre < CONFIG$min_n || n_war < CONFIG$min_n) {
    cat("  Skipping — pre_war:", n_pre, "war:", n_war, "\n")
    return(NULL)
  }

  covars_sector <- keep_varying_covars(cv$all, df_sector)
  out_dir <- paste0("matched_outputs/", kind, "/", weapon,
                    "/time_by_sector/", sector)

  tryCatch(
    run_matching(
      df              = df_sector,
      group_col       = "war_period",
      targets_cols    = TARGETS,
      output_dir      = out_dir,
      treatment       = "war",
      control         = "pre_war",
      caliper_ps_sd   = CONFIG$caliper_ps,
      match_mode      = CONFIG$match_mode,
      all_covars      = covars_sector,
      fact_covars     = cv$fact,
      demo_covars     = cv$demo,
      binary_targets  = BINARY_TARGETS,
      interaction_var = "war_period"
      # war_interaction left FALSE: war_period IS the treatment here
    ),
    error = function(e) cat("  ERROR:", conditionMessage(e), "\n")
  )

  res_path <- file.path(out_dir, "match_logit", "results.csv")
  if (!file.exists(res_path)) return(NULL)
  read_csv(res_path, show_col_types = FALSE) %>%
    mutate(kind = kind, weapon = weapon, sector = sector)
}

# =============================================================================
# MAIN LOOP
# =============================================================================
all_results <- list()

for (kind in CONFIG$kinds) {
  for (weapon in CONFIG$weapons) {
    df <- load_weapon_df(kind, weapon)
    cv <- drop_personal_covars(covars_for(df), kind)

    cat("\n====", kind, "/", weapon, "====\n")
    print(df %>% count(ethnic, war_period) %>% arrange(ethnic, war_period))

    for (sector in CONFIG$sectors) {
      df_sector <- df %>% filter(ethnic == sector)
      res <- run_sector(df_sector, weapon, sector, cv, kind)
      if (!is.null(res)) all_results[[paste(kind, weapon, sector, sep = "_")]] <- res
    }
  }
}

# =============================================================================
# SUMMARY TABLE
# =============================================================================
if (length(all_results) == 0) {
  stop("No results collected — check that run_matching succeeded for at least one combination.")
}

summary_table <- bind_rows(all_results) %>%
  select(any_of(c("kind","weapon","sector","outcome","is_binary",
                  "mean_treatment","mean_control",
                  "diff","ci_low","ci_high","p_value",
                  "d_cohens","n_treatment_unique","n_control_unique"))) %>%
  arrange(kind, weapon, outcome, sector)

print(summary_table)
for (k in unique(summary_table$kind))
  write_csv(filter(summary_table, kind == k),
            paste0("matched_outputs/", k, "/time_by_sector_weapons_summary.csv"))

# Robustness – alternative matching for significant district / time findings (SM, Section 3.3.1)

In [ ]:
# =============================================================================
# ROBUSTNESS — ALTERNATIVE MATCHING FOR SIGNIFICANT DISTRICT / TIME FINDINGS
# Same data and covariates as the primary one-vs-all district and time-period
# analyses; only the matching algorithm changes (Gower distance, exact on offense vars).
# =============================================================================
ROBUST <- list(
  kind          = "impute",
  districts     = list(possession_only = c("South")),
  periods       = list(possession_only = c("corona_period", "war_period")),
  match_specs   = list(list(mode = "gower",     only_fact = FALSE),
                       list(mode = "exact_cem", only_fact = TRUE)),
  control_label = "All_Others",
  caliper_q     = 0.9,
  caliper_ps    = 0.2,
  gower_demo_w  = 1,
  gower_fact_w  = 3,
  bonferroni_k  = 30,
  near_constant_thresh = 0.95,
  war_start     = as.Date("2023-10-07"),
  corona_start  = as.Date("2020-03-01")
)

ROBUST_MULTILABEL_COLS <- c("weapon_motive", "weapon_use", "weapon_storage")
ROBUST_WEAPON_COUNT_CANDIDATES <- c(
  "handgun", "submachine_gun", "assault_rifle", "sniper_rifle", "hunting_rifle",
  "improvised_firearm", "grenade", "explosive_charge", "molotov",
  "anti_tank_missile", "ammunition_only", "weapon_parts", "unidentified_firearm"
)
ROBUST_PERIOD_LABELS <- list(
  corona_period = list(control = "pre_corona", treatment = "post_corona"),
  war_period    = list(control = "pre_war",    treatment = "post_war")
)

robust_expand_multilabel <- function(df, col) {
  tags     <- lapply(strsplit(df[[col]], ","), stringr::str_squish)
  all_tags <- sort(unique(unlist(tags)))
  all_tags <- all_tags[!is.na(all_tags) & all_tags != ""]
  for (tag in all_tags)
    df[[paste0(col, "_", tag)]] <- as.integer(sapply(tags, function(x) tag %in% x))
  df[[col]] <- NULL
  df
}

robust_load_weapon <- function(kind, weapon) {
  df <- read_csv(paste0("data/", kind, "/", weapon, ".csv"),
                 locale = locale(encoding = "UTF-8"), show_col_types = FALSE) %>%
    filter(Criminal_record != "לא ידוע") %>%
    mutate(
      district           = str_squish(as.character(district)),
      age                = as.numeric(age),
      is_actual_sentence = as.integer(is_actual_sentence),
      Date               = dmy(Date),
      war_period         = if_else(Date >= ROBUST$war_start, 1L, 0L),
      corona_period      = if_else(Date >= ROBUST$corona_start, 1L, 0L)
    ) %>%
    filter(ethnic %in% c("Arab", "Jewish")) %>%
    filter(!(judge_ethnic %in% c("Anonymous", "Unknown")))
  for (col in ROBUST_MULTILABEL_COLS)
    if (col %in% names(df)) df <- robust_expand_multilabel(df, col)
  df
}

robust_count_cols <- function(df) {
  ROBUST_WEAPON_COUNT_CANDIDATES[sapply(ROBUST_WEAPON_COUNT_CANDIDATES, function(v) {
    if (!v %in% names(df)) return(FALSE)
    max(table(df[[v]])) / nrow(df) <= ROBUST$near_constant_thresh
  })]
}

# Covariates as in the primary analyses: district is excluded when it is the
# group (district analysis) and included for the time-period analysis.
robust_weapon_covars <- function(df, with_district) {
  count_cols <- robust_count_cols(df)
  fact <- c("actual_damage","premeditation",
            "escape_attempt","has_independent_offenses","undercover_sale",
            "magazines","bullets", count_cols)
  demo <- c("age","marital_status","year","ethnic",
            "regret","confession","Court","Criminal_record","judge_ethnic")
  if (with_district) demo <- c("district", setdiff(demo, "year"))
  list(all = c(demo, fact), fact = fact, demo = demo)
}

robust_run <- function(df, group_col, treatment, control, out_dir, cv,
                       interaction_var = "ethnic", bonferroni_k = 1) {
  for (spec in ROBUST$match_specs) {
    cat("\n===", out_dir, "|", spec$mode,
        if (spec$only_fact) "(offense vars only)" else "", "===\n")
    tryCatch(
      run_matching(
        df = df, group_col = group_col,
        targets_cols = c("min_punishment", "max_punishment", "is_actual_sentence"),
        output_dir = out_dir, treatment = treatment, control = control,
        caliper_q = ROBUST$caliper_q, caliper_ps_sd = ROBUST$caliper_ps,
        match_mode = spec$mode,
        all_covars = cv$all, fact_covars = cv$fact, demo_covars = cv$demo,
        gower_demo_weight = ROBUST$gower_demo_w, gower_fact_weight = ROBUST$gower_fact_w,
        only_fact_vars = spec$only_fact,
        binary_targets = "is_actual_sentence",
        interaction_var = interaction_var,
        bonferroni_k = bonferroni_k
      ),
      error = function(e) cat("  ERROR:", conditionMessage(e), "\n")
    )
  }
}

# --- District one-vs-all ---
for (weapon in names(ROBUST$districts)) {
  df <- robust_load_weapon(ROBUST$kind, weapon)
  cv <- robust_weapon_covars(df, with_district = FALSE)
  for (target_district in ROBUST$districts[[weapon]]) {
    df_temp <- df %>%
      mutate(district_one_vs_all = if_else(district == target_district,
                                           target_district, ROBUST$control_label))
    robust_run(df_temp, "district_one_vs_all", target_district, ROBUST$control_label,
               out_dir = paste0("matched_outputs/", ROBUST$kind, "/", weapon, "/district_",
                                gsub(" ", "_", target_district), "_vs_All_Others"),
               cv = cv, bonferroni_k = ROBUST$bonferroni_k)
  }
}

# --- Time periods ---
for (weapon in names(ROBUST$periods)) {
  df <- robust_load_weapon(ROBUST$kind, weapon)
  cv <- robust_weapon_covars(df, with_district = TRUE)
  for (period_col in ROBUST$periods[[weapon]]) {
    lbl       <- ROBUST_PERIOD_LABELS[[period_col]]
    group_col <- paste0(period_col, "_group")
    df_local  <- df %>%
      mutate(!!group_col := if_else(.data[[period_col]] == 1, lbl$treatment, lbl$control))
    robust_run(df_local, group_col, lbl$treatment, lbl$control,
               out_dir = file.path("matched_outputs", ROBUST$kind, weapon,
                                   gsub("_period$", "", period_col)),
               cv = cv,
               interaction_var = if (period_col == "war_period") "war_period" else "ethnic")
  }
}